# Azure Key Vault Secrets Sync with Terraform and SPN

Terraform creates the resource group, Key Vault, application, service principal, client secret, RBAC assignments, Vault destination and test association.

> The SPN client secret is sensitive and is stored in Terraform state. Do not commit or share the state or plan files.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")

subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
tenant_id = os.environ["AZURE_TENANT_ID"]
os.environ["ARM_SUBSCRIPTION_ID"] = subscription_id
os.environ["ARM_TENANT_ID"] = tenant_id
os.environ["TF_VAR_azure_subscription_id"] = subscription_id


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((p / ".env" for p in (Path.cwd(), *Path.cwd().parents) if (p / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find .env")
load_dotenv(ENV_FILE)

## Authenticate and confirm the Azure subscription

In [ ]:
! az login --tenant $ARM_TENANT_ID --subscription $ARM_SUBSCRIPTION_ID
! az account show --query '{subscription:name,subscriptionId:id,tenantId:tenantId}' --output table
! az provider register --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --wait
! az provider show --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --query '{namespace:namespace,state:registrationState}' --output table

## Initialize and validate

In [ ]:
! terraform -chdir=terraform-azure-spn init
! terraform -chdir=terraform-azure-spn validate

## Review the plan

In [ ]:
! terraform -chdir=terraform-azure-spn plan

## Apply

The configuration waits 60 seconds after creating the application password and RBAC assignment to allow Microsoft Entra propagation.

In [ ]:
! terraform -chdir=terraform-azure-spn apply -auto-approve

## Verify Vault and Azure Key Vault

In [ ]:
! vault read sys/sync/destinations/azure-kv/mapfre-spn-azure-kv
! vault read -format=json sys/sync/destinations/azure-kv/mapfre-spn-azure-kv/associations | jq

In [ ]:
%%bash
KEY_VAULT_NAME=$(terraform -chdir=terraform-azure-spn output -raw key_vault_name)
SECRET_NAME=$(terraform -chdir=terraform-azure-spn output -raw external_secret_name)
az keyvault secret show \
  --vault-name "$KEY_VAULT_NAME" \
  --name "$SECRET_NAME" \
  --query '{name:name,enabled:attributes.enabled,updated:attributes.updated}' \
  --output json

# CLEAN UP

In [ ]:
! terraform -chdir=terraform-azure-spn destroy -auto-approve